# Cuaderno de Trabajo — FraudShield AI

Bitácora del proyecto: decisiones de arquitectura, conceptos aprendidos, errores encontrados, glosario y diario de trabajo.

Este notebook se actualiza al final de cada etapa del proyecto.

## Índice

1. [Decisiones de arquitectura](#decisiones)
2. [Conceptos nuevos aprendidos](#conceptos)
3. [Errores encontrados y cómo se resolvieron](#errores)
4. [Ideas para futuras mejoras](#ideas)
5. [Preguntas abiertas](#preguntas)
6. [Métricas por versión del modelo](#metricas)
7. [Glosario técnico](#glosario)
8. [Diario de trabajo](#diario)
9. [Lecciones aprendidas por etapa](#lecciones)

<a id='decisiones'></a>
## 1. Decisiones de arquitectura

| # | Decisión | Justificación |
|---|---|---|
| 1 | Arquitectura conceptual de detección en cascada (2 etapas: filtro rápido + análisis profundo) | Refleja cómo funcionan sistemas reales de fraude (Stripe, PayPal, bancos) |
| 2 | El proyecto implementa completamente solo la Etapa 2 (análisis profundo) | El valor del portafolio está en ciencia de datos aplicada, no en ingeniería de datos en tiempo real (streaming/Kafka) |
| 3 | Métrica de negocio: minimizar **pérdida neta** (fraude no detectado + fricción por falsos positivos), no maximizar detección de fraude a cualquier costo | El costo de un falso negativo es mucho mayor al de un falso positivo, pero un exceso de falsos positivos también tiene costo real (fricción al cliente) |
| 4 | Dataset elegido: **IEEE-CIS Fraud Detection** (Kaggle/Vesta), no el dataset clásico de tarjetas de crédito (ULB) | El dataset ULB viene con features ya transformadas por PCA (anónimas), lo cual elimina la posibilidad de hacer feature engineering real. IEEE-CIS tiene columnas con significado de negocio y datos sucios reales |
| 5 | Se usa únicamente `train_transaction.csv` y `train_identity.csv`; se descartan los archivos `test_*` | Los archivos de test no incluyen `isFraud` (son para la competencia original de Kaggle). El proyecto hace su propio split train/test |
| 6 | Entorno de desarrollo: WSL2 + Ubuntu + Python 3.11 (no la versión 3.14 del sistema) + entorno virtual (`venv`) | Compatibilidad del ecosistema ML (XGBoost, LightGBM, CatBoost, SHAP, MLflow) con versiones estables de Python; WSL2 facilita Docker más adelante |
| 7 | Estrategia para columnas con nulidad idéntica (bloque V138-V162) | Una sola feature binaria `is_missing` por grupo funcional, no una por columna, para evitar redundancia |

<a id='conceptos'></a>
## 2. Conceptos nuevos aprendidos (con mis palabras)

> Completar cada concepto con tu propia explicación, no copiar la definición formal.

**Costo asimétrico de errores:**
> _(tu explicación aquí)_

**Missing Not At Random (MNAR):**
> _(tu explicación aquí)_

**Feature Scaling (Normalización / Estandarización):**
> _(tu explicación aquí)_

**Por qué los modelos basados en árboles no necesitan escalado:**
> _(tu explicación aquí)_

**Cardinalidad de una variable categórica:**
> _(tu explicación aquí)_

<a id='errores'></a>
## 3. Errores encontrados y cómo se resolvieron

| Error | Causa | Solución |
|---|---|---|
| Terminal "trabada" tras correr `jupyter notebook --no-browser` | El comando corre el servidor en primer plano, ocupando la terminal indefinidamente | `Ctrl+C` para detener el servidor; se optó por usar VS Code + extensión Jupyter en su lugar |
| `ls -lh data/raw` mostró "total 0" | Se ejecutó antes de que terminara la descarga/descompresión, o desde la carpeta incorrecta | Verificar con `ls` en la carpeta raíz del proyecto primero, luego repetir en `data/raw` |
| Token de Kaggle pegado en texto plano en el chat | Desconocimiento de que cualquier texto compartido debe tratarse como potencialmente expuesto | Revocar el token comprometido y generar uno nuevo; nunca compartir credenciales en texto plano |
| Confusión entre `KGAT_...` (token nuevo) y formato legacy `kaggle.json` | Kaggle mantiene dos sistemas de autenticación en paralelo | Se usó el token nuevo guardado en `~/.kaggle/access_token` |

<a id='ideas'></a>
## 4. Ideas para futuras mejoras (sin implementar todavía)

- Explorar si existen más bloques de nulidad idéntica entre las ~339 columnas `V`, no solo el bloque V138-V162 ya confirmado.
- Evaluar downcasting de tipos numéricos (`float64` → `float32`) para reducir el uso de memoria (~1.7 GB actual en `train_transaction`).
- Considerar `RobustScaler` en vez de `StandardScaler` para el modelo base, dado que en fraude existen outliers reales (no solo ruido).

<a id='preguntas'></a>
## 5. Preguntas abiertas

- ¿Cómo se manejará exactamente el join entre `train_transaction` y `train_identity`, dado que solo ~24% de las transacciones tienen identidad asociada? (pendiente para etapa de pipeline/feature engineering)
- ¿Qué agrupación se usará para `P_emaildomain` / `R_emaildomain` dado su cardinalidad media (~60 valores)?

<a id='metricas'></a>
## 6. Métricas por versión del modelo

> Se completará a partir de la Etapa 4 (modelo base), cuando existan resultados que registrar.

| Versión | Modelo | Precision | Recall | F1 | PR-AUC | Notas |
|---|---|---|---|---|---|---|
| — | — | — | — | — | — | — |

<a id='glosario'></a>
## 7. Glosario técnico

- **MNAR (Missing Not At Random):** patrón de datos faltantes donde la ausencia del valor está relacionada con la variable objetivo u otras variables, no ocurre al azar.
- **Feature Scaling:** transformar variables numéricas a una escala/rango comparable, sin alterar la información que contienen.
- **PR-AUC:** área bajo la curva Precision-Recall; métrica preferida sobre ROC-AUC en problemas con fuerte desbalance de clases.
- **Cardinalidad:** número de valores únicos distintos que toma una variable categórica.
- **Data leakage:** cuando información de fuera del conjunto de entrenamiento (p. ej. datos de test) influye indebidamente en el entrenamiento del modelo, inflando artificialmente su desempeño.

<a id='diario'></a>
## 8. Diario de trabajo

### Sesión 1 — Etapas 1 y 2

**Qué hice:**
- Definí el problema de negocio de FraudShield AI (arquitectura en cascada, costo asimétrico, métrica de pérdida neta).
- Configuré el entorno de desarrollo completo: WSL2, Ubuntu, Python 3.11, entorno virtual, VS Code conectado a WSL.
- Configuré la API de Kaggle y descargué el dataset IEEE-CIS Fraud Detection.
- Hice la primera inspección cruda de `train_transaction` y `train_identity` (shape, dtypes, nulidad, variable objetivo, columnas categóricas).

**Qué aprendí:**
- A distinguir problema de negocio de problema técnico antes de tocar código.
- El concepto de MNAR y por qué la ausencia de un dato puede ser una señal válida.
- Cómo detectar bloques de columnas con nulidad idéntica usando correlación de máscaras booleanas.

**Qué haré después:**
- Etapa 3: Análisis Exploratorio de Datos completo.

<a id='lecciones'></a>
## 9. Lecciones aprendidas por etapa

### Etapa 1 — Definición del problema
- Un problema de ML mal definido invalida todo el trabajo posterior, sin importar qué tan bueno sea el modelo.
- Es fácil caer en "resume-driven development" (elegir tecnología para verse bien en el CV en vez de porque el problema lo requiere) — hay que nombrarlo cuando pasa.

### Etapa 2 — Comprensión del dataset
- El dataset elegido determina qué habilidades se pueden demostrar realmente (un dataset pre-procesado con PCA elimina la posibilidad de feature engineering genuino).
- La nulidad no es solo "un problema a limpiar": puede ser señal, y puede tener estructura de grupo (varias columnas fallando juntas) en vez de ser independiente columna por columna.